# 06 — Table gold
Une ligne par transaction : 32 features (passé strict) + étiquette `is_fraud` + colonne `split`.
Exclusions anti-fuite : erreurs de la transaction courante, attributs « photo 2020 »
(`year_pin_last_changed`, `expires_month`, `cards_issued`), pays (segmentation uniquement).

In [ ]:
import sys
import time
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SILVER = ROOT / "data" / "processed" / "silver"
FEATURES = ROOT / "data" / "processed" / "features"
GOLD = ROOT / "data" / "processed" / "gold" / "gold.parquet"
TMP_LINUX = Path.home() / ".cache" / "secumon-duckdb-tmp"
TMP_LINUX.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET temp_directory = '{TMP_LINUX.as_posix()}'")
con.execute("SET memory_limit = '3GB'")
con.execute("SET threads = 4")
con.execute("SET preserve_insertion_order = false")
con.execute("SET enable_progress_bar = false")

print("Dossier :", Path.cwd())
for f in ["transactions", "customers", "cards"]:
    print(f"silver/{f}.parquet :", (SILVER / f"{f}.parquet").exists())
for f in ["velocity", "context"]:
    print(f"features/{f}.parquet :", (FEATURES / f"{f}.parquet").exists())

In [ ]:
import importlib
from secumon.ml.features import gold
importlib.reload(gold)

t0 = time.perf_counter()
n = gold.build_gold(con, SILVER, FEATURES, GOLD)
print(f"{n} lignes ecrites en {time.perf_counter() - t0:.0f} s")
print(f"Taille : {GOLD.stat().st_size / 1e6:.1f} Mo")
print(f"Features : {len(gold.FEATURE_COLUMNS)} ({len(gold.NUMERIC_FEATURES)} numeriques, "
      f"{len(gold.CATEGORICAL_FEATURES)} categorielles)")

In [ ]:
import importlib
from secumon.ml.features import gold
importlib.reload(gold)
print("Correction presente dans le module charge :", "CASE WHEN date_diff" in gold.GOLD_SQL)

In [ ]:
con.sql(f"""
    SELECT split,
           COUNT(*)                     AS transactions,
           SUM(is_fraud::INT)::BIGINT   AS fraudes
    FROM '{GOLD}'
    GROUP BY split
    ORDER BY split
""").df()

In [ ]:
manquants = ",\n".join(f"COUNT(*) FILTER (WHERE {c} IS NULL) AS {c}" for c in gold.FEATURE_COLUMNS)
display(con.sql(f"SELECT {manquants} FROM '{GOLD}'").df().T.rename(columns={0: "valeurs_manquantes"}))

con.sql(f"""
    SELECT MIN(age_at_tx)                                     AS age_min,
           MAX(age_at_tx)                                     AS age_max,
           COUNT(*) FILTER (WHERE account_age_months < 0)     AS compte_ouvert_apres_tx,
           MIN(account_age_months)                            AS anciennete_min_mois
    FROM '{GOLD}'
""").df().T

In [ ]:
import importlib
from secumon.ml.features import gold
importlib.reload(gold)
print("Module corrige charge :", "CASE WHEN date_diff" in gold.GOLD_SQL)

t0 = time.perf_counter()
n = gold.build_gold(con, SILVER, FEATURES, GOLD)
print(f"{n} lignes ecrites en {time.perf_counter() - t0:.0f} s")

con.sql(f"""
    SELECT COUNT(*) FILTER (WHERE account_age_months IS NULL)   AS anciennete_manquante,
           COUNT(*) FILTER (WHERE account_age_months < 0)       AS anciennete_negative,
           MIN(account_age_months)                              AS anciennete_min_mois,
           MIN(age_at_tx)                                       AS age_min,
           MAX(age_at_tx)                                       AS age_max
    FROM '{GOLD}'
""").df().T

## Conclusion
`gold.parquet` : 24 386 900 lignes, 32 features (28 numériques, 4 catégorielles),
construite en ~75 s par `secumon/ml/features/gold.py` (4 tests).

| Jeu | Transactions | Fraudes |
|---|---|---|
| train (1991–2014) | 15 471 192 | 18 064 |
| validation (2015) | 1 701 371 | 3 281 |
| test A — connu (2016) | 1 708 924 | 3 579 |
| test B — campagne (2017–2020) | 5 505 413 | 4 833 |

**Valeurs manquantes voulues** (= « pas d'information ») : en ligne (2 720 821, pas de lieu),
premier achat (6 139 cartes / 2 000 clients), historique insuffisant, lieu imprécis.

**Anomalie corrigée** : 1 496 transactions antérieures à l'ouverture du compte (photo 2020,
jusqu'à −21 mois) → ancienneté mise à NULL au lieu d'une valeur négative (test de non-régression).

**Garantie testée** : aucune feature ne contient l'étiquette, le pays, un identifiant
ni une colonne « photo 2020 » postérieure à la transaction.